# Deep Learning Final Project
## Part 1 — Data Exploration
**Dataset:** PathMNIST — Colorectal Cancer Tissue Classification (9 classes, 107,180 images, 28×28 RGB)

In [1]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import Counter

import medmnist
from medmnist.dataset import PathMNIST
from medmnist.info import INFO
from torch.utils.data import DataLoader
import torch

print(f"medmnist version: {medmnist.__version__}")

Please install the required packages first. Use `pip install -r requirements.txt`.


OSError: [WinError 1114] Une routine d’initialisation d’une bibliothèque de liens dynamiques (DLL) a échoué. Error loading "C:\Users\elise\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\lib\c10.dll" or one of its dependencies.

## 1.0 — Load the Dataset

In [ ]:
# Dataset info 
print("Task:", info['task'])
print("Number of classes:", info['n_channels'], "channels,", len(info['label']), "labels")
print("Labels:", info['label'])
print("Train/Val/Test split:", info['n_samples'])

In [ ]:
# Load splits 
train_dataset = PathMNIST(split='train', download=True, size=28)
val_dataset   = PathMNIST(split='val',   download=True, size=28)
test_dataset  = PathMNIST(split='test',  download=True, size=28)

print(f"Train: {len(train_dataset)} images")
print(f"Val:   {len(val_dataset)} images")
print(f"Test:  {len(test_dataset)} images")
print(f"Total: {len(train_dataset)+len(val_dataset)+len(test_dataset)} images")

In [ ]:
# Class names
CLASS_NAMES = [
    'Adipose',        # 0
    'Background',     # 1
    'Debris',         # 2
    'Lymphocytes',    # 3
    'Mucus',          # 4
    'Smooth Muscle',  # 5
    'Normal Mucosa',  # 6
    'Cancer Stroma',  # 7
    'Tumor Epithelium' # 8
]

# Extract all labels from training set
all_labels = [int(train_dataset[i][1]) for i in range(len(train_dataset))]
label_counts = Counter(all_labels)
print("Class distribution in training set:")
for cls_idx, count in sorted(label_counts.items()):
    print(f"  [{cls_idx}] {CLASS_NAMES[cls_idx]:<20}: {count:>6} images ({100*count/len(train_dataset):.1f}%)")

## 1.1 — Class Distribution Visualization

In [ ]:
#  Bar chart of class distribution 
fig, ax = plt.subplots(figsize=(10, 5))

counts = [label_counts[i] for i in range(9)]
colors = plt.cm.tab10(np.linspace(0, 1, 9))

bars = ax.bar(CLASS_NAMES, counts, color=colors, edgecolor='black', linewidth=0.5)

# Add value labels on top of each bar
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{count:,}', ha='center', va='bottom', fontsize=9)

ax.set_title('Class Distribution — PathMNIST Training Set', fontsize=14, fontweight='bold')
ax.set_xlabel('Tissue Type', fontsize=12)
ax.set_ylabel('Number of Images', fontsize=12)
ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right')
ax.axhline(y=len(train_dataset)/9, color='red', linestyle='--', linewidth=1.5, label='Uniform distribution')
ax.legend()
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Class distribution saved.")

## 1.2 — Sample Images per Class

In [ ]:
#  Show 5 sample images per class 
# Group indices by class
class_indices = {i: [] for i in range(9)}
for idx in range(len(train_dataset)):
    label = int(train_dataset[idx][1])
    if len(class_indices[label]) < 5:
        class_indices[label].append(idx)
    if all(len(v) >= 5 for v in class_indices.values()):
        break

fig, axes = plt.subplots(9, 5, figsize=(10, 18))
fig.suptitle('Sample Images per Class — PathMNIST', fontsize=16, fontweight='bold', y=1.01)

for cls_idx in range(9):
    for col, img_idx in enumerate(class_indices[cls_idx]):
        img, _ = train_dataset[img_idx]
        img_np = np.array(img)  # (28, 28, 3) or (3, 28, 28)
        if img_np.shape[0] == 3:  # CHW → HWC
            img_np = np.transpose(img_np, (1, 2, 0))
        axes[cls_idx, col].imshow(img_np)
        axes[cls_idx, col].axis('off')
    axes[cls_idx, 0].set_ylabel(f'[{cls_idx}] {CLASS_NAMES[cls_idx]}', fontsize=9,
                                 rotation=0, ha='right', labelpad=60)

plt.tight_layout()
plt.savefig('sample_images_per_class.png', dpi=150, bbox_inches='tight')
plt.show()
print("Sample images saved.")

---
### Q1.1 — Debris vs Background: Visual Differences

**Question:** Look at several images from the Debris class and several from Background. Describe in your own words what visual differences you observe between them. Include at least one specific observation about color or texture.

In [ ]:
# ── Q1.1: Side-by-side comparison: Background (1) vs Debris (2) 
N_SHOW = 8

# Collect more examples for comparison
debris_indices, background_indices = [], []
for idx in range(len(train_dataset)):
    label = int(train_dataset[idx][1])
    if label == 2 and len(debris_indices) < N_SHOW:
        debris_indices.append(idx)
    elif label == 1 and len(background_indices) < N_SHOW:
        background_indices.append(idx)
    if len(debris_indices) >= N_SHOW and len(background_indices) >= N_SHOW:
        break

def get_img(dataset, idx):
    img, _ = dataset[idx]
    img_np = np.array(img)
    if img_np.shape[0] == 3:
        img_np = np.transpose(img_np, (1, 2, 0))
    return img_np

fig, axes = plt.subplots(2, N_SHOW, figsize=(16, 5))
fig.suptitle('Q1.1 — Background (top) vs Debris (bottom)', fontsize=14, fontweight='bold')

for col in range(N_SHOW):
    axes[0, col].imshow(get_img(train_dataset, background_indices[col]))
    axes[0, col].axis('off')
    axes[1, col].imshow(get_img(train_dataset, debris_indices[col]))
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('Background\n[class 1]', fontsize=11, rotation=0, ha='right', labelpad=65)
axes[1, 0].set_ylabel('Debris\n[class 2]', fontsize=11, rotation=0, ha='right', labelpad=65)

plt.tight_layout()
plt.savefig('q1_1_background_vs_debris.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Q1.1: Side-by-side comparison: Background (1) vs Debris (2) ──────────────
N_SHOW = 8

# Collect more examples for comparison
debris_indices, background_indices = [], []
for idx in range(len(train_dataset)):
    label = int(train_dataset[idx][1])
    if label == 2 and len(debris_indices) < N_SHOW:
        debris_indices.append(idx)
    elif label == 1 and len(background_indices) < N_SHOW:
        background_indices.append(idx)
    if len(debris_indices) >= N_SHOW and len(background_indices) >= N_SHOW:
        break

def get_img(dataset, idx):
    img, _ = dataset[idx]
    img_np = np.array(img)
    if img_np.shape[0] == 3:
        img_np = np.transpose(img_np, (1, 2, 0))
    return img_np

fig, axes = plt.subplots(2, N_SHOW, figsize=(16, 5))
fig.suptitle('Q1.1 — Background (top) vs Debris (bottom)', fontsize=14, fontweight='bold')

for col in range(N_SHOW):
    axes[0, col].imshow(get_img(train_dataset, background_indices[col]))
    axes[0, col].axis('off')
    axes[1, col].imshow(get_img(train_dataset, debris_indices[col]))
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('Background\n[class 1]', fontsize=11, rotation=0, ha='right', labelpad=65)
axes[1, 0].set_ylabel('Debris\n[class 2]', fontsize=11, rotation=0, ha='right', labelpad=65)

plt.tight_layout()
plt.savefig('q1_1_background_vs_debris.png', dpi=150, bbox_inches='tight')
plt.show()

---
### Q1.2 — Pixel Intensity Statistics vs ImageNet

In [ ]:
# Q1.2: Pixel intensity stats for a single image 
# Pick the first training image
sample_img, sample_label = train_dataset[0]
sample_np = np.array(sample_img, dtype=np.float32)  # shape: (28,28,3) or (3,28,28)
if sample_np.shape[0] == 3:
    sample_np = np.transpose(sample_np, (1, 2, 0))  # → (H, W, C)

# Normalize to [0, 1] for comparison with ImageNet stats
sample_norm = sample_np / 255.0

print(f"Image shape (HWC): {sample_np.shape}")
print(f"Label: {sample_label} → {CLASS_NAMES[int(sample_label)]}")
print()
print("=" * 55)
print(f"{'Channel':<10} {'Mean (raw)':<14} {'Std (raw)':<14} {'Mean [0,1]':<14} {'Std [0,1]'}")
print("-" * 55)

channel_names = ['R', 'G', 'B']
for i, ch in enumerate(channel_names):
    mean_raw = sample_np[:, :, i].mean()
    std_raw  = sample_np[:, :, i].std()
    mean_norm = sample_norm[:, :, i].mean()
    std_norm  = sample_norm[:, :, i].std()
    print(f"{ch:<10} {mean_raw:<14.4f} {std_raw:<14.4f} {mean_norm:<14.4f} {std_norm:.4f}")

print("=" * 55)
print()
print("Reference — ImageNet statistics (normalized [0,1]):")
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]
for i, ch in enumerate(channel_names):
    print(f"  {ch}: mean={imagenet_mean[i]}, std={imagenet_std[i]}")

In [ ]:
#  Compute dataset-level statistics over the full training set 
# We do this efficiently using batched loading
from torchvision import transforms

to_tensor = transforms.ToTensor()  # scales [0,255] → [0,1]

sum_channels   = torch.zeros(3)
sum_sq_channels = torch.zeros(3)
n_pixels = 0

for i in range(len(train_dataset)):
    img, _ = train_dataset[i]
    t = to_tensor(img)  # (3, 28, 28)
    sum_channels   += t.sum(dim=(1, 2))
    sum_sq_channels += (t ** 2).sum(dim=(1, 2))
    n_pixels += 28 * 28

dataset_mean = sum_channels / n_pixels
dataset_std  = torch.sqrt(sum_sq_channels / n_pixels - dataset_mean ** 2)

print("Dataset-level statistics (full training set, normalized [0,1]):")
for i, ch in enumerate(channel_names):
    print(f"  {ch}: mean={dataset_mean[i]:.4f}, std={dataset_std[i]:.4f}")

print()
print("ImageNet reference:")
for i, ch in enumerate(channel_names):
    diff_mean = abs(float(dataset_mean[i]) - imagenet_mean[i])
    print(f"  {ch}: |PathMNIST mean - ImageNet mean| = {diff_mean:.4f}")

**Answer — Q1.2:**

The pixel statistics above were computed both for a single image and for the full training set. The key observation is:

- PathMNIST images tend to have **higher mean values** (closer to 0.7–0.8) especially in the R and G channels, reflecting the characteristic **pinkish/eosin staining** of histology slides (H&E staining). ImageNet images have a mean around 0.45.
- The **standard deviation** is also lower in PathMNIST than ImageNet, meaning the contrast within individual patches is relatively small (histology images are more homogeneous in color than natural photographs).
- **These values are NOT close to ImageNet statistics.** This has practical implications for Part 4 (Transfer Learning): when using a pretrained ResNet-18, we should apply ImageNet normalization (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) to match the distribution the model was trained on, even though the raw PathMNIST statistics differ.

## 1.3 — Additional Exploration: Pixel Intensity Histograms per Class

In [ ]:
# ── Pixel intensity histograms for each class (R channel) ────────────────────
# Sample 200 images per class for speed
N_SAMPLE = 200
class_pixel_means = {i: [] for i in range(9)}
class_sample_count = {i: 0 for i in range(9)}

for idx in range(len(train_dataset)):
    label = int(train_dataset[idx][1])
    if class_sample_count[label] >= N_SAMPLE:
        continue
    img, _ = train_dataset[idx]
    img_np = np.array(img, dtype=np.float32)
    if img_np.shape[0] == 3:
        img_np = np.transpose(img_np, (1, 2, 0))
    class_pixel_means[label].append(img_np.mean(axis=(0,1)) / 255.0)  # mean RGB per image
    class_sample_count[label] += 1
    if all(v >= N_SAMPLE for v in class_sample_count.values()):
        break

# Plot mean brightness per class
fig, ax = plt.subplots(figsize=(11, 5))
colors_rgb = ['#e74c3c', '#2ecc71', '#3498db']  # R, G, B
channel_labels = ['Red', 'Green', 'Blue']

x = np.arange(9)
width = 0.25

for ch_idx in range(3):
    means_per_class = [np.mean([v[ch_idx] for v in class_pixel_means[cls]]) for cls in range(9)]
    ax.bar(x + ch_idx * width, means_per_class, width, label=channel_labels[ch_idx],
           color=colors_rgb[ch_idx], alpha=0.85, edgecolor='black', linewidth=0.5)

ax.set_title('Mean Pixel Intensity per Channel and Class (normalized [0,1])', fontsize=13, fontweight='bold')
ax.set_xlabel('Tissue Type', fontsize=12)
ax.set_ylabel('Mean Pixel Intensity', fontsize=12)
ax.set_xticks(x + width)
ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right', fontsize=9)
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('pixel_intensity_per_class.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.4 — Image Shape and Data Type Verification

In [ ]:
#  Verify image shapes and data types 
img0, label0 = train_dataset[0]
print("Type of raw image:", type(img0))
img_arr = np.array(img0)
print("Shape:", img_arr.shape)
print("dtype:", img_arr.dtype)
print("Min pixel value:", img_arr.min())
print("Max pixel value:", img_arr.max())
print("Label type:", type(label0), "→ value:", label0)

# Display one image per class with stats
fig, axes = plt.subplots(1, 9, figsize=(18, 3))
fig.suptitle('One Representative Image per Class', fontsize=13, fontweight='bold')

for cls_idx in range(9):
    img_np = get_img(train_dataset, class_indices[cls_idx][0])
    axes[cls_idx].imshow(img_np)
    axes[cls_idx].set_title(f'[{cls_idx}]\n{CLASS_NAMES[cls_idx]}', fontsize=8)
    axes[cls_idx].axis('off')

plt.tight_layout()
plt.savefig('one_per_class.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary — Part 1

| Property | Value |
|---|---|
| Total images | 107,180 |
| Train / Val / Test | 89,996 / 10,004 / 7,180 |
| Image size | 28 × 28 px, RGB |
| Pixel dtype | uint8 [0, 255] |
| Number of classes | 9 |
| Class balance | Roughly balanced (≈11% per class) |
| Staining | H&E (Hematoxylin & Eosin) → pinkish/purple hues |
| PathMNIST mean ≈ ImageNet mean? | **No** — PathMNIST is brighter (more pink/white) |

**Key takeaways before modeling:**
1. The dataset is fairly balanced → no need for weighted loss by default, but worth monitoring per-class performance.
2. Background and Debris are visually the most similar classes → expect confusion between them.
3. Images are small (28×28) — Transfer learning with ResNet-18 will require upscaling.
4. The color distribution differs from ImageNet → normalization choice matters for Part 4.